# Dark Manifold V31: Steady-State Glycolysis

**Key Fix**: V29.5 and V30 crashed because they were closed systems — glucose depleted, ATP crashed.

**V31 adds**:
1. **Glucose uptake** — continuous influx simulating cellular transport
2. **Lactate export** — removal to prevent accumulation
3. **NAD⁺ regeneration** — mitochondrial electron transport (simplified)

This creates an **open system** that can reach true steady-state with oscillations.

## Expected Behavior
| V30 (closed) | V31 (open) |
|--------------|------------|
| ATP crashes to 0 | ATP maintains ~2-3 mM |
| Glucose depletes | Glucose steady ~1-2 mM |
| Energy charge → 0 | Energy charge ~0.85 |
| No oscillations | Glycolytic oscillations possible |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import time
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## 1. Open Metabolic Network with Boundary Fluxes

In [ ]:
class OpenGlycolysisNetwork:
    """
    Open glycolysis network with:
    - Glucose uptake (GLUT transporters)
    - Lactate export (MCT transporters)
    - Simplified mitochondrial NAD+ regeneration
    """
    
    def __init__(self):
        self.metabolites = [
            # Glycolysis
            'glc', 'g6p', 'f6p', 'fbp', 'gap', 'bpg', '3pg', '2pg', 'pep', 'pyr',
            # End products
            'lac', 'acoa',
            # Energy
            'atp', 'adp', 'amp',
            # Redox
            'nad', 'nadh',
            # Phosphate
            'pi'
        ]
        self.n_met = len(self.metabolites)
        self.met_idx = {m: i for i, m in enumerate(self.metabolites)}
        
        # Indices for easy reference
        self.glc_idx = self.met_idx['glc']
        self.atp_idx = self.met_idx['atp']
        self.adp_idx = self.met_idx['adp']
        self.nad_idx = self.met_idx['nad']
        self.nadh_idx = self.met_idx['nadh']
        self.lac_idx = self.met_idx['lac']
        self.pyr_idx = self.met_idx['pyr']
        
        # Reactions with BOUNDARY FLUXES
        # Format: (name, substrates, products, kcat, Km, is_boundary)
        self.reactions = [
            # === BOUNDARY FLUXES (open system) ===
            # Glucose uptake (GLUT1/4) - constant supply
            ('GLUT', [], [(0, 1)], 2.0, 1.0, True),  # 2 mM/min glucose influx
            # Lactate export (MCT1/4)
            ('MCT', [(10, 1)], [], 5.0, 2.0, True),  # Lactate efflux
            # Mitochondrial NAD+ regeneration (simplified ETC)
            ('ETC', [(16, 1)], [(15, 1)], 10.0, 0.1, True),  # NADH → NAD+
            
            # === GLYCOLYSIS ===
            ('HK', [(0, 1), (12, 1)], [(1, 1), (13, 1)], 100.0, 0.1, False),
            ('PGI', [(1, 1)], [(2, 1)], 600.0, 0.3, False),
            ('PFK', [(2, 1), (12, 1)], [(3, 1), (13, 1)], 100.0, 0.05, False),
            ('ALDO', [(3, 1)], [(4, 2)], 40.0, 0.05, False),
            ('GAPDH', [(4, 1), (15, 1), (17, 1)], [(5, 1), (16, 1)], 200.0, 0.05, False),
            ('PGK', [(5, 1), (13, 1)], [(6, 1), (12, 1)], 400.0, 0.3, False),
            ('PGM', [(6, 1)], [(7, 1)], 500.0, 0.5, False),
            ('ENO', [(7, 1)], [(8, 1)], 300.0, 0.3, False),
            ('PK', [(8, 1), (13, 1)], [(9, 1), (12, 1)], 200.0, 0.2, False),
            
            # === PYRUVATE FATE ===
            ('LDH', [(9, 1), (16, 1)], [(10, 1), (15, 1)], 300.0, 0.2, False),
            ('PDH', [(9, 1), (15, 1)], [(11, 1), (16, 1)], 20.0, 0.1, False),
            
            # === ATP HOMEOSTASIS ===
            ('ATPase', [(12, 1)], [(13, 1), (17, 1)], 80.0, 1.0, False),  # Reduced from 150
            ('ADK', [(12, 1), (14, 1)], [(13, 2)], 500.0, 0.5, False),
        ]
        self.n_rxn = len(self.reactions)
        
        # Build stoichiometry matrix
        self.S = np.zeros((self.n_met, self.n_rxn))
        self.kcat = np.zeros(self.n_rxn)
        self.Km = np.zeros(self.n_rxn)
        self.is_boundary = np.zeros(self.n_rxn, dtype=bool)
        
        for j, rxn in enumerate(self.reactions):
            name, subs, prods, kcat, km, is_boundary = rxn
            self.kcat[j] = kcat
            self.Km[j] = km
            self.is_boundary[j] = is_boundary
            for idx, coef in subs:
                self.S[idx, j] -= coef
            for idx, coef in prods:
                self.S[idx, j] += coef
        
        # Initial concentrations (mM) - physiological
        self.M0 = np.array([
            2.0,   # glc - lower, will be replenished
            0.08,  # g6p
            0.03,  # f6p  
            0.02,  # fbp
            0.04,  # gap
            0.002, # bpg
            0.06,  # 3pg
            0.01,  # 2pg
            0.02,  # pep
            0.08,  # pyr
            1.0,   # lac
            0.02,  # acoa
            3.0,   # atp
            0.5,   # adp
            0.1,   # amp
            0.8,   # nad - slightly lower to allow equilibration
            0.3,   # nadh - slightly higher
            5.0,   # pi
        ])

network = OpenGlycolysisNetwork()
print(f"Network: {network.n_met} metabolites, {network.n_rxn} reactions")
print(f"Boundary fluxes: {sum(network.is_boundary)} (glucose uptake, lactate export, NAD+ regen)")

## 2. Model with Open System Dynamics

In [ ]:
class OpenSystemModel(nn.Module):
    """Model with boundary fluxes for open-system steady state."""
    
    def __init__(self, network):
        super().__init__()
        
        self.n_met = network.n_met
        self.n_rxn = network.n_rxn
        
        self.register_buffer('S', torch.tensor(network.S, dtype=torch.float32))
        self.register_buffer('kcat', torch.tensor(network.kcat, dtype=torch.float32))
        self.register_buffer('Km', torch.tensor(network.Km, dtype=torch.float32))
        self.register_buffer('M0', torch.tensor(network.M0, dtype=torch.float32))
        self.register_buffer('is_boundary', torch.tensor(network.is_boundary))
        
        # Build substrate mask
        substrate_mask = torch.zeros(self.n_rxn, self.n_met)
        for j, rxn in enumerate(network.reactions):
            _, subs, _, _, _, _ = rxn
            for idx, _ in subs:
                substrate_mask[j, idx] = 1.0
        self.register_buffer('substrate_mask', substrate_mask)
        
        # Concentration bounds
        self.min_conc = 1e-9
        self.max_conc = 50.0
    
    def compute_flux(self, M):
        """Compute reaction fluxes with proper boundary handling."""
        batch_size = M.shape[0]
        v = torch.zeros(batch_size, self.n_rxn, device=M.device)
        
        for j in range(self.n_rxn):
            if self.is_boundary[j]:
                # Boundary flux: simple mass action or constant
                if self.substrate_mask[j].sum() == 0:
                    # Pure input (e.g., glucose uptake) - constant rate
                    v[:, j] = self.kcat[j]
                else:
                    # Output depends on substrate (e.g., lactate export)
                    sub_indices = (self.substrate_mask[j] > 0).nonzero(as_tuple=True)[0]
                    if len(sub_indices) > 0:
                        sub_conc = M[:, sub_indices[0]]
                        v[:, j] = self.kcat[j] * sub_conc / (self.Km[j] + sub_conc)
            else:
                # Internal reaction: Michaelis-Menten
                sub_indices = (self.substrate_mask[j] > 0).nonzero(as_tuple=True)[0]
                if len(sub_indices) == 0:
                    v[:, j] = self.kcat[j]
                else:
                    # Product of saturation terms
                    sat = torch.ones(batch_size, device=M.device)
                    for idx in sub_indices:
                        sat = sat * M[:, idx] / (self.Km[j] + M[:, idx] + 1e-10)
                    v[:, j] = self.kcat[j] * sat
        
        return v
    
    def forward(self, M, dt):
        """Single Euler step."""
        M = torch.clamp(M, min=self.min_conc)
        v = self.compute_flux(M)
        dM_dt = torch.matmul(v, self.S.T)
        M_next = M + dM_dt * dt
        return torch.clamp(M_next, min=self.min_conc, max=self.max_conc), v
    
    @torch.no_grad()
    def simulate(self, n_steps, dt, save_every=1000, progress_fn=None):
        """Run simulation."""
        M = self.M0.unsqueeze(0).to(self.S.device)
        
        n_save = n_steps // save_every + 1
        M_hist = torch.zeros(n_save, self.n_met, device=M.device)
        v_hist = torch.zeros(n_save, self.n_rxn, device=M.device)
        M_hist[0] = M[0]
        
        save_idx = 0
        for step in range(n_steps):
            M, v = self.forward(M, dt)
            
            if (step + 1) % save_every == 0:
                save_idx += 1
                if save_idx < n_save:
                    M_hist[save_idx] = M[0]
                    v_hist[save_idx] = v[0]
                
                if progress_fn and (save_idx % 2000 == 0):
                    progress_fn(step + 1, n_steps, M[0], v[0])
        
        return M_hist, v_hist

model = OpenSystemModel(network).to(device)
print(f"Model ready on {device}")

## 3. Benchmark

In [ ]:
# Benchmark
print("Benchmarking...")

# Warmup
M = model.M0.unsqueeze(0).to(device)
with torch.no_grad():
    for _ in range(2000):
        M, _ = model.forward(M, 1e-6)

# Timed
if device.type == 'cuda':
    torch.cuda.synchronize()
start = time.time()

test_steps = 100_000
M = model.M0.unsqueeze(0).to(device)
with torch.no_grad():
    for _ in range(test_steps):
        M, _ = model.forward(M, 1e-6)

if device.type == 'cuda':
    torch.cuda.synchronize()
elapsed = time.time() - start

steps_per_sec = test_steps / elapsed
print(f"Speed: {steps_per_sec:,.0f} steps/second")

## 4. Run Steady-State Simulation

In [ ]:
# Configuration
REAL_TIME_BUDGET_MIN = 2  # Real minutes
VIRTUAL_DURATION_MIN = 20  # Virtual minutes
SAVE_EVERY = 500

budget_sec = REAL_TIME_BUDGET_MIN * 60
n_steps = int(steps_per_sec * budget_sec * 0.85)
dt_min = VIRTUAL_DURATION_MIN / n_steps
dt_us = dt_min * 60 * 1_000_000

print("="*70)
print("DARK MANIFOLD V31 - OPEN SYSTEM STEADY STATE")
print("="*70)
print(f"Real time: {REAL_TIME_BUDGET_MIN} min | Virtual: {VIRTUAL_DURATION_MIN} min")
print(f"Steps: {n_steps:,} | Resolution: {dt_us:.1f} µs")
print("="*70)

start_time = time.time()
def progress_fn(step, total, M, v):
    elapsed = time.time() - start_time
    pct = 100 * step / total
    atp = M[network.atp_idx].item()
    glc = M[network.glc_idx].item()
    lac = M[network.lac_idx].item()
    print(f"  {pct:5.1f}% | ATP:{atp:.2f} | Glc:{glc:.2f} | Lac:{lac:.2f} mM")

print("\nRunning...")
M_history, v_history = model.simulate(
    n_steps=n_steps,
    dt=dt_min,
    save_every=SAVE_EVERY,
    progress_fn=progress_fn
)

total_time = time.time() - start_time
print(f"\n✅ Complete in {total_time:.1f}s")

## 5. Visualization

In [ ]:
# Convert to numpy
M = M_history.cpu().numpy()
V = v_history.cpu().numpy()
time_min = np.arange(len(M)) * SAVE_EVERY * dt_min

# Extract key metabolites
atp = M[:, network.atp_idx]
adp = M[:, network.adp_idx]
amp = M[:, network.met_idx['amp']]
glc = M[:, network.glc_idx]
lac = M[:, network.lac_idx]
pyr = M[:, network.pyr_idx]
nad = M[:, network.nad_idx]
nadh = M[:, network.nadh_idx]

# Computed quantities
ec = (atp + 0.5 * adp) / (atp + adp + amp + 0.01)
redox = nad / (nad + nadh + 0.01)

print(f"Data: {len(M)} points over {time_min[-1]:.1f} min")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# ATP/ADP/AMP
ax = axes[0, 0]
ax.plot(time_min, atp, 'b-', label='ATP', lw=1)
ax.plot(time_min, adp, 'r-', label='ADP', lw=1)
ax.plot(time_min, amp, 'g-', label='AMP', lw=1)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Adenine Nucleotide Pool')
ax.legend()
ax.grid(True, alpha=0.3)

# Energy Charge
ax = axes[0, 1]
ax.plot(time_min, ec, 'purple', lw=1)
ax.axhline(0.85, color='g', ls='--', alpha=0.7, label='Optimal')
ax.axhline(0.7, color='orange', ls='--', alpha=0.7, label='Stress')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Energy Charge')
ax.set_title('Cellular Energy Charge')
ax.set_ylim([0.4, 1.0])
ax.legend()
ax.grid(True, alpha=0.3)

# Redox State
ax = axes[0, 2]
ax.plot(time_min, nad, 'b-', label='NAD⁺', lw=1)
ax.plot(time_min, nadh, 'r-', label='NADH', lw=1)
ax2 = ax.twinx()
ax2.plot(time_min, redox, 'g--', label='Ratio', lw=1, alpha=0.7)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax2.set_ylabel('NAD⁺ Ratio', color='g')
ax.set_title('Redox State')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Glucose/Lactate
ax = axes[1, 0]
ax.plot(time_min, glc, 'b-', label='Glucose', lw=1)
ax.plot(time_min, lac, 'orange', label='Lactate', lw=1)
ax.plot(time_min, pyr, 'g-', label='Pyruvate', lw=1)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Glycolysis: Substrate & Products')
ax.legend()
ax.grid(True, alpha=0.3)

# Key Fluxes
ax = axes[1, 1]
glut_flux = V[:, 0]  # Glucose uptake
hk_flux = V[:, 3]    # Hexokinase
pk_flux = V[:, 11]   # Pyruvate kinase
mct_flux = V[:, 1]   # Lactate export
ax.plot(time_min, glut_flux, 'b-', label='GLUT (uptake)', lw=1)
ax.plot(time_min, hk_flux, 'r-', label='HK', lw=1)
ax.plot(time_min, pk_flux, 'g-', label='PK', lw=1)
ax.plot(time_min, mct_flux, 'orange', label='MCT (export)', lw=1)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Flux (mM/min)')
ax.set_title('Key Metabolic Fluxes')
ax.legend()
ax.grid(True, alpha=0.3)

# Summary
ax = axes[1, 2]
ax.axis('off')
summary = f"""
V31 STEADY-STATE RESULTS
{'='*40}

Resolution:    {dt_us:.1f} µs
Duration:      {VIRTUAL_DURATION_MIN} virtual min
Compute:       {total_time:.1f} real sec

FINAL STATE
{'-'*40}
ATP:           {atp[-1]:.3f} mM
ADP:           {adp[-1]:.3f} mM
Energy Charge: {ec[-1]:.3f}
Glucose:       {glc[-1]:.3f} mM
Lactate:       {lac[-1]:.3f} mM
NAD⁺/NADH:     {redox[-1]:.3f}

STEADY STATE?
{'-'*40}
ATP σ (last 50%): {np.std(atp[len(atp)//2:]):.4f} mM
EC σ (last 50%):  {np.std(ec[len(ec)//2:]):.4f}
"""
ax.text(0.1, 0.9, summary, transform=ax.transAxes, fontsize=10,
        family='monospace', va='top')

plt.tight_layout()
plt.savefig('v31_steady_state.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Frequency Analysis (Now Should Show Real Oscillations)

In [ ]:
# Only analyze if we have steady state (ATP not crashed)
if atp[-1] > 0.5:
    print("System reached steady state - analyzing oscillations...")
    
    # Use second half of data (after transient)
    half = len(M) // 2
    
    fs = 1 / (SAVE_EVERY * dt_us * 1e-6)  # Hz
    print(f"Sampling frequency: {fs:.2f} Hz")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for ax, (met_key, met_name) in zip(axes.flat, [('atp', 'ATP'), ('nadh', 'NADH'), ('pyr', 'Pyruvate'), ('glc', 'Glucose')]):
        data = M[half:, network.met_idx[met_key]]
        data_detrended = data - gaussian_filter1d(data, sigma=len(data)//20)
        
        nperseg = min(len(data)//4, 10000)
        f, psd = signal.welch(data_detrended, fs, nperseg=nperseg)
        
        ax.semilogy(f, psd)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('PSD')
        ax.set_title(f'{met_name} Fluctuation Spectrum (steady state)')
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, min(fs/2, 10)])
    
    plt.tight_layout()
    plt.savefig('v31_frequency_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"WARNING: ATP crashed to {atp[-1]:.3f} mM - no steady state reached")
    print("Possible issues:")
    print("  - Glucose uptake rate too low")
    print("  - ATPase consumption too high")
    print("  - Need to tune boundary flux rates")

## 7. Save Results

In [ ]:
output = {
    'metadata': {
        'model': 'Dark Manifold V31',
        'description': 'Open system with glucose uptake and lactate export',
        'virtual_duration_min': VIRTUAL_DURATION_MIN,
        'real_time_sec': total_time,
        'resolution_us': dt_us,
        'n_metabolites': network.n_met,
        'n_reactions': network.n_rxn,
        'boundary_fluxes': ['GLUT (glucose uptake)', 'MCT (lactate export)', 'ETC (NAD+ regen)'],
    },
    'final_state': {
        'atp': float(atp[-1]),
        'adp': float(adp[-1]),
        'amp': float(amp[-1]),
        'energy_charge': float(ec[-1]),
        'glucose': float(glc[-1]),
        'lactate': float(lac[-1]),
        'nad': float(nad[-1]),
        'nadh': float(nadh[-1]),
        'redox_ratio': float(redox[-1]),
    },
    'steady_state_check': {
        'atp_std_last_half': float(np.std(atp[len(atp)//2:])),
        'ec_std_last_half': float(np.std(ec[len(ec)//2:])),
        'is_steady': bool(atp[-1] > 0.5 and np.std(atp[len(atp)//2:]) < 0.1),
    },
    'metabolite_names': network.metabolites,
    'time_min': time_min.tolist(),
    'metabolites': {name: M[:, i].tolist() for name, i in network.met_idx.items()},
}

with open('dark_manifold_v31_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"✅ Saved to dark_manifold_v31_results.json")

## 8. Interpretation

### What V31 Should Show (if working correctly):

1. **Stable ATP** around 2-3 mM (not crashing to 0)
2. **Energy charge** maintaining 0.8-0.9 (not dropping to 0)
3. **Glucose** at steady-state (uptake = consumption)
4. **Lactate** at steady-state (production = export)

### If Still Crashing:
Tune these parameters:
- Increase `GLUT kcat` (glucose uptake rate)
- Decrease `ATPase kcat` (cellular ATP consumption)
- Adjust `ETC kcat` (NAD+ regeneration)

### Glycolytic Oscillations:
At steady-state, we might see oscillations due to:
- PFK allosteric regulation (not yet implemented)
- ATP/ADP ratio feedback
- NAD+/NADH cycling

These would appear as peaks in the frequency analysis at ~0.1-1 Hz.